In [ ]:
# ------------------------------------------------------------
# Test 3: Adaptive Beams + Vectorized Postprocessing COMBINED
# Expected: 35.1 or 35.2 (micro-gains might stack!)
# ------------------------------------------------------------

import os
os.environ["OMP_NUM_THREADS"] = "4"
os.environ["MKL_NUM_THREADS"] = "4"
os.environ["CUDA_LAUNCH_BLOCKING"] = "0"
os.environ["TORCH_CUDNN_V8_API_ENABLED"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "true"

import re
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader, Sampler
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from tqdm.auto import tqdm

# ============================================================
# CONFIG
# ============================================================

CONFIG = {
    "test_path": "/kaggle/input/deep-past-initiative-machine-translation/test.csv",
    "model_path": "/kaggle/input/final-byt5/byt5-akkadian-optimized-34x",
    "max_length": 512,
    "batch_size": 8,
    "num_beams": 8,
    "max_new_tokens": 512,
    "length_penalty": 1.3,
    "early_stopping": True,
    "num_buckets": 4,
    "device": torch.device("cuda" if torch.cuda.is_available() else "cpu")
}

print("="*60)
print("TEST 3: ADAPTIVE BEAMS + VECTORIZED POSTPROC")
print("="*60)
print(f"Device: {CONFIG['device']}")
print(f"✓ Adaptive beams: ENABLED")
print(f"✓ Vectorized postproc: ENABLED")
print("="*60)

# ============================================================
# PREPROCESSING
# ============================================================

def preprocess_input(text):
    if pd.isna(text):
        return ""
    text = str(text)
    text = re.sub(r'(\.{3,}|…+|……)', '<big_gap>', text)
    text = re.sub(r'(xx+|\s+x\s+)', '<gap>', text)
    return text

# ============================================================
# VECTORIZED POSTPROCESSOR
# ============================================================

class VectorizedPostprocessor:
    def __init__(self):
        self.patterns = {
            "gap": re.compile(r"(\[x\]|\(x\)|\bx\b)", re.I),
            "big_gap": re.compile(r"(\.{3,}|…|\[\.+\])"),
            "annotations": re.compile(r"\((fem|plur|pl|sing|singular|plural|\?|!)\.?\s*\w*\)", re.I),
            "repeated_words": re.compile(r"\b(\w+)(?:\s+\1\b)+"),
            "whitespace": re.compile(r"\s+"),
            "punct_space": re.compile(r"\s+([.,:])"),
            "repeated_punct": re.compile(r"([.,])\1+"),
        }
        self.subscript_trans = str.maketrans("₀₁₂₃₄₅₆₇₈₉", "0123456789")
        self.special_chars_trans = str.maketrans("ḫḪ", "hH")
        self.forbidden_chars = '!?()"—–<>⌈⌋⌊[]+ʾ/;'
        self.forbidden_trans = str.maketrans("", "", self.forbidden_chars)
    
    def postprocess_batch(self, translations):
        """Vectorized batch postprocessing using pandas"""
        s = pd.Series(translations)
        valid_mask = s.apply(lambda x: isinstance(x, str) and bool(x.strip()))
        s.loc[~valid_mask] = ""
        
        s = s.str.translate(self.special_chars_trans)
        s = s.str.translate(self.subscript_trans)
        s = s.str.replace(self.patterns["gap"], "<gap>", regex=True)
        s = s.str.replace(self.patterns["big_gap"], "<big_gap>", regex=True)
        s = s.str.replace("<gap> <gap>", "<big_gap>", regex=False)
        s = s.str.replace("<big_gap> <big_gap>", "<big_gap>", regex=False)
        s = s.str.replace(self.patterns["annotations"], "", regex=True)
        
        s = s.str.replace("<gap>", "\x00GAP\x00", regex=False)
        s = s.str.replace("<big_gap>", "\x00BIG\x00", regex=False)
        s = s.str.translate(self.forbidden_trans)
        s = s.str.replace("\x00GAP\x00", " <gap> ", regex=False)
        s = s.str.replace("\x00BIG\x00", " <big_gap> ", regex=False)
        
        s = s.str.replace(r"(\d+)\.5\b", r"\1 ½", regex=True)
        s = s.str.replace(r"\b0\.5\b", "½", regex=True)
        s = s.str.replace(r"(\d+)\.25\b", r"\1 ¼", regex=True)
        s = s.str.replace(r"\b0\.25\b", "¼", regex=True)
        s = s.str.replace(r"(\d+)\.75\b", r"\1 ¾", regex=True)
        s = s.str.replace(r"\b0\.75\b", "¾", regex=True)
        s = s.str.replace(r"(\d+)\.33+\d*\b", r"\1 ⅓", regex=True)
        s = s.str.replace(r"\b0\.33+\d*\b", "⅓", regex=True)
        s = s.str.replace(r"(\d+)\.66+\d*\b", r"\1 ⅔", regex=True)
        s = s.str.replace(r"\b0\.66+\d*\b", "⅔", regex=True)
        
        s = s.str.replace(self.patterns["repeated_words"], r"\1", regex=True)
        for n in range(4, 1, -1):
            pattern = r"\b((?:\w+\s+){" + str(n-1) + r"}\w+)(?:\s+\1\b)+"
            s = s.str.replace(pattern, r"\1", regex=True)
        
        s = s.str.replace(self.patterns["punct_space"], r"\1", regex=True)
        s = s.str.replace(self.patterns["repeated_punct"], r"\1", regex=True)
        s = s.str.replace(self.patterns["whitespace"], " ", regex=True)
        s = s.str.strip().str.strip("-").str.strip()
        
        return s.tolist()

# ============================================================
# ADAPTIVE BEAM FUNCTION
# ============================================================

def get_adaptive_beam_size(attention_mask, base_beams=8):
    """
    Adaptive beam search:
    - Short sequences (<100 tokens): Use 4 beams
    - Long sequences (>=100 tokens): Use 8 beams
    """
    lengths = attention_mask.sum(dim=1)
    short_beams = max(4, base_beams // 2)
    if lengths[0] < 100:
        return short_beams
    else:
        return base_beams

# ============================================================
# BUCKET BATCH SAMPLER
# ============================================================

class BucketBatchSampler(Sampler):
    def __init__(self, dataset, batch_size, num_buckets):
        self.dataset = dataset
        self.batch_size = batch_size
        
        lengths = [len(text.split()) for _, text in dataset]
        sorted_indices = sorted(range(len(lengths)), key=lambda i: lengths[i])
        
        bucket_size = max(1, len(sorted_indices) // max(1, num_buckets))
        self.buckets = []
        for i in range(num_buckets):
            start = i * bucket_size
            end = None if i == num_buckets - 1 else (i + 1) * bucket_size
            self.buckets.append(sorted_indices[start:end])
    
    def __iter__(self):
        for bucket in self.buckets:
            for i in range(0, len(bucket), self.batch_size):
                yield bucket[i : i + self.batch_size]
    
    def __len__(self):
        total = 0
        for bucket in self.buckets:
            total += (len(bucket) + self.batch_size - 1) // self.batch_size
        return total

# ============================================================
# DATASET
# ============================================================

class AkkadianDataset(Dataset):
    def __init__(self, dataframe):
        self.sample_ids = dataframe['id'].tolist()
        self.input_texts = [
            "translate Akkadian to English: " + preprocess_input(text)
            for text in dataframe['transliteration']
        ]
    
    def __len__(self):
        return len(self.sample_ids)
    
    def __getitem__(self, index):
        return self.sample_ids[index], self.input_texts[index]

# ============================================================
# MAIN INFERENCE
# ============================================================

print("\nLoading test data...")
test_df = pd.read_csv(CONFIG['test_path'])
print(f"✓ {len(test_df)} samples")

print("\nLoading model...")
model = AutoModelForSeq2SeqLM.from_pretrained(CONFIG['model_path'])
model = model.to(CONFIG['device']).eval()
tokenizer = AutoTokenizer.from_pretrained(CONFIG['model_path'])
print(f"✓ Model loaded")

postprocessor = VectorizedPostprocessor()
print("✓ Postprocessor ready")

dataset = AkkadianDataset(test_df)
batch_sampler = BucketBatchSampler(dataset, CONFIG['batch_size'], CONFIG['num_buckets'])

def collate_fn(batch):
    ids = [s[0] for s in batch]
    texts = [s[1] for s in batch]
    tokenized = tokenizer(texts, max_length=CONFIG['max_length'], 
                         padding=True, truncation=True, return_tensors='pt')
    return ids, tokenized

dataloader = DataLoader(dataset, batch_sampler=batch_sampler, 
                       collate_fn=collate_fn, num_workers=2)

print(f"✓ DataLoader ready: {len(dataloader)} batches")
print("\nGenerating translations...")
results = []

with torch.inference_mode():
    for batch_ids, tokenized in tqdm(dataloader, desc="Translating"):
        input_ids = tokenized.input_ids.to(CONFIG['device'])
        attention_mask = tokenized.attention_mask.to(CONFIG['device'])
        
        # ADAPTIVE BEAMS
        num_beams = get_adaptive_beam_size(attention_mask, CONFIG['num_beams'])
        
        outputs = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            num_beams=num_beams,
            max_new_tokens=CONFIG['max_new_tokens'],
            length_penalty=CONFIG['length_penalty'],
            early_stopping=CONFIG['early_stopping']
        )
        
        translations = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        
        # VECTORIZED POSTPROCESSING
        cleaned = postprocessor.postprocess_batch(translations)
        
        results.extend(zip(batch_ids, cleaned))

submission = pd.DataFrame(results, columns=['id', 'translation'])
submission.to_csv('submission.csv', index=False)

print("\n" + "="*60)
print("✓ DONE!")
print("="*60)
print(f"Translations: {len(submission)}")
print(f"Optimizations: Adaptive beams + Vectorized postproc")
print(f"\nSample outputs:")
for i in range(min(3, len(submission))):
    print(f"  {i}: {submission.iloc[i]['translation'][:80]}...")
print("="*60)